# Pipeline d'Analyse Exploratoire — MINST

Exploration visuelle

Premier coup d'oeil certain 1 ressemble à des 7 et certaine 4 à des 9

La moyene globale des pixels et plutot bass 33.2 donc le data set est majoritairement sombre
La variance globale est assez élevé donc les image sont assez contrasté
Et les distribution suivent des lois normale autour de la moyenne donc la majorité des image ont un contraste similaire et un niveau de luminosité similaire

PAs d'outlier 

Les images sont noires avec des zones blanches bien marquées


In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np

# Chargement du dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# 🔥 Pré-calcul des indices pour éviter les np.where répétés
indices_par_digit = {d: np.where(y_train == d)[0] for d in range(10)}

# 🎨 Affichage optimisé (moins lourd)
fig, axes = plt.subplots(10, 5, figsize=(6, 12))

for digit in range(10):
    idxs = indices_par_digit[digit][:5]
    
    for j, idx in enumerate(idxs):
        axes[digit, j].imshow(x_train[idx], cmap='gray')
        axes[digit, j].axis('off')
        
        if j == 0:
            axes[digit, j].set_title(f"{digit}")

# ❌ On enlève tight_layout (lent)
# plt.tight_layout()

plt.show()


# 📊 Stats globales
print(f"Mean globale : {x_train.mean():.2f}")
print(f"Std globale  : {x_train.std():.2f}")



x_flat = x_train.reshape(len(x_train), -1)
means = x_flat.mean(axis=1)
stds  = x_flat.std(axis=1)

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.hist(means, bins=30)
plt.title("Moyennes")

plt.subplot(1,2,2)
plt.hist(stds, bins=30)
plt.title("Écarts-types")

plt.show()

## Preprocessing

Le data set est propre pas d'outliers, le data set est hompgène pas besoin de standardisation. On va juste normaliser

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np

# Chargement du dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalisation
x_train = x_train / 255.0
x_test  = x_test  / 255.0

# Vérification
print(f"Min: {x_train.min()}")   # → 0.0
print(f"Max: {x_train.max()}")   # → 1.0
print(f"Mean: {x_train.mean():.4f}")  # → ~0.13 (était ~33)

## Clustering non supervisé

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import numpy as np

N = 5000

# PREPARATION
def preparer_donnees(x):
    x_flat = x.reshape(len(x), -1)
    pca50 = PCA(n_components=50, random_state=42)
    x_50d = pca50.fit_transform(x_flat)
    pca2 = PCA(n_components=2, random_state=42)
    x_2d = pca2.fit_transform(x_flat)
    return x_50d, x_2d

# t-SNE
def reduire_tsne(x_50d, n=N):
    print(f't-SNE en cours sur {n} points (~30s)...')
    tsne = TSNE(n_components=2, random_state=42)
    return tsne.fit_transform(x_50d[:n])

# K-DISTANCE
def plot_kdistance(x, k=5, n=N):
    x_sub = x[:n]
    nbrs = NearestNeighbors(n_neighbors=k).fit(x_sub)
    distances, _ = nbrs.kneighbors(x_sub)
    k_dist = np.sort(distances[:, -1])
    plt.figure(figsize=(8, 4))
    plt.plot(k_dist)
    plt.xlabel('Points tries')
    plt.ylabel(f'Distance au {k}e voisin')
    plt.title(f'k-distance graph (k={k}) sur t-SNE')
    plt.grid(True)
    plt.show()
    print(f'Min : {k_dist.min():.2f} | Mediane : {np.median(k_dist):.2f} | Max : {k_dist.max():.2f}')

# KMEANS
def clustering_kmeans(x_50d):
    km = KMeans(n_clusters=10, random_state=42, n_init=10)
    return km.fit_predict(x_50d)

# DBSCAN sur t-SNE
def clustering_dbscan(x_tsne, eps, n=N):
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(x_tsne[:n])
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_outliers = np.sum(labels == -1)
    print(f'DBSCAN -> {n_clusters} clusters | {n_outliers} outliers sur {n} points')
    return labels

# COULEURS DISTINCTES
def make_colors(labels):
    palette = list(plt.cm.tab10.colors) + list(plt.cm.Set2.colors)
    color_map = {}
    cluster_idx = 0
    for u in sorted(set(labels)):
        if u == -1:
            color_map[u] = (0.75, 0.75, 0.75)
        else:
            color_map[u] = palette[cluster_idx % len(palette)]
            cluster_idx += 1
    return np.array([color_map[l] for l in labels])

# VISUALISATION
def visualiser_clusters(x_2d, x_tsne, y_true, labels, titre_model, n=N):
    n_labels = len(labels)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'Clustering - {titre_model} ({n_clusters} clusters)', fontsize=15, fontweight='bold')
    axes[0, 0].scatter(x_2d[:, 0], x_2d[:, 1], c=y_true, cmap='tab10', s=1, alpha=0.5)
    axes[0, 0].set_title('PCA - Vraies classes')
    axes[0, 1].scatter(x_2d[:n_labels, 0], x_2d[:n_labels, 1], c=make_colors(labels), s=1, alpha=0.5)
    axes[0, 1].set_title(f'PCA - {titre_model}')
    axes[1, 0].scatter(x_tsne[:, 0], x_tsne[:, 1], c=y_true[:n], cmap='tab10', s=5, alpha=0.8)
    axes[1, 0].set_title('t-SNE - Vraies classes')
    axes[1, 1].scatter(x_tsne[:, 0], x_tsne[:, 1], c=make_colors(labels[:n]), s=5, alpha=0.8)
    axes[1, 1].set_title(f't-SNE - {titre_model}')
    plt.tight_layout()
    plt.show()

# EXECUTION
x_50d, x_2d = preparer_donnees(x_train)
x_tsne = reduire_tsne(x_50d)

labels_km = clustering_kmeans(x_50d)
visualiser_clusters(x_2d, x_tsne, y_train, labels_km, 'KMeans')

plot_kdistance(x_tsne, k=5, n=N)

labels_db = clustering_dbscan(x_tsne, eps=3)
visualiser_clusters(x_2d, x_tsne, y_train, labels_db, 'DBSCAN')


## Regression Logistique

In [ ]:
# =============================================================
# OBJECTIF : Tester une regression logistique sur MNIST
# C'est le modele le plus simple possible pour la classification.
# Il traite chaque image comme un vecteur de 784 pixels et cherche
# une frontiere lineaire entre les 10 classes de chiffres.
# On s'en sert comme baseline : si le CNN fait mieux, c'est justifie.
# =============================================================

import numpy as np
import keras
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import time

# Chargement du dataset (on recharge pour que la cellule soit autonome)
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalisation [0,255] -> [0,1] ET aplatissement (28x28) -> vecteur (784,)
# Pourquoi normaliser : la LR est sensible a l'echelle des valeurs,
# sans ca le gradient diverge et la convergence est tres lente
# Pourquoi aplatir : la LR ne comprend pas les images 2D,
# elle attend un tableau (n_samples, n_features)
x_train_flat = (x_train / 255.0).reshape(len(x_train), -1)
x_test_flat  = (x_test  / 255.0).reshape(len(x_test),  -1)

# On note le temps pour le comparer au CNN plus tard
t0 = time.time()

# Creation du modele
# max_iter=1000 : autorise 1000 iterations pour converger (MNIST en a besoin)
# solver='saga'  : algorithme optimise pour les grands datasets (60 000 images)
# n_jobs=-1      : utilise tous les coeurs CPU disponibles, va plus vite
# random_state=42: resultats reproductibles entre deux executions
logreg = LogisticRegression(max_iter=1000, solver='saga', n_jobs=-1, random_state=42)

# Entrainement : le modele apprend les poids optimaux sur les 60 000 images
logreg.fit(x_train_flat, y_train)

# Enregistrement du temps total d'entrainement
train_time_lr = time.time() - t0

# Prediction sur le test set (10 000 images jamais vues pendant l'entrainement)
# Pourquoi le test set : mesurer la capacite de generalisation du modele
y_pred_lr = logreg.predict(x_test_flat)

# Accuracy : proportion d'images correctement classees
acc_lr = (y_pred_lr == y_test).mean()

print(f'Accuracy : {acc_lr:.4f}')
print(f'Temps entrainement : {train_time_lr:.1f}s')
print()

# Rapport detaille : precision, rappel, F1-score pour chaque chiffre
# Pourquoi : l'accuracy seule cache les differences entre chiffres,
# certains sont bien plus difficiles (ex: 4 vs 9, 3 vs 8)
print(classification_report(y_test, y_pred_lr))

# Matrice de confusion : lignes = vrais labels, colonnes = predictions
# Une case hors diagonale = une confusion frequente entre deux chiffres
cm = confusion_matrix(y_test, y_pred_lr)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm).plot(ax=ax, colorbar=False)
ax.set_title('Matrice de confusion - Regression Logistique')
plt.tight_layout()
plt.show()

## CNN (Keras)

In [ ]:
# =============================================================
# OBJECTIF : Entrainer un CNN (Convolutional Neural Network) sur MNIST
# Contrairement a la regression logistique qui voit les pixels
# independamment, le CNN detecte des patterns spatiaux : bords, courbes,
# angles. C'est pourquoi il est bien plus performant sur les images.
# Architecture : 2 blocs Conv+Pool pour extraire les features,
# puis des couches Dense pour la classification finale.
# =============================================================

import numpy as np
import keras
from keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import time

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalisation + ajout d'un axe de canal
# Pourquoi [..., np.newaxis] : Keras attend des images de forme (28, 28, 1)
# Le '1' = nombre de canaux (1 pour niveaux de gris, 3 pour RGB)
# On NE fait PAS de reshape en vecteur : le CNN travaille sur la structure 2D
x_train_cnn = (x_train / 255.0)[..., np.newaxis]
x_test_cnn  = (x_test  / 255.0)[..., np.newaxis]

# Construction du modele avec l'API Sequential
# Pourquoi Sequential : on empile les couches une par une
model = keras.Sequential([

    # Forme de l'entree : images 28x28 avec 1 canal
    layers.Input(shape=(28, 28, 1)),

    # 1er bloc : 32 filtres convolutifs 3x3, activation ReLU
    # Pourquoi Conv2D : detecte des patterns locaux (bords, courbes, traits)
    # chaque filtre apprend a reconnaitre un pattern different dans l'image
    # Pourquoi ReLU : met les valeurs negatives a 0, introduit la non-linearite
    # necessaire pour apprendre des fonctions complexes
    layers.Conv2D(32, (3, 3), activation='relu'),

    # MaxPooling 2x2 : garde la valeur max dans chaque fenetre 2x2
    # Pourquoi : reduit la taille de moitie, moins de parametres,
    # et rend le modele robuste aux petits decalages de l'image
    layers.MaxPooling2D((2, 2)),

    # 2eme bloc : 64 filtres, detecte des patterns plus complexes
    # Pourquoi plus de filtres : les couches profondes combinent les patterns
    # simples du 1er bloc pour former des formes plus abstraites
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Flatten : convertit la carte 2D en vecteur 1D
    # Pourquoi : les couches Dense qui suivent attendent un vecteur 1D
    layers.Flatten(),

    # Dense 128 : combine les features extraites pour distinguer les chiffres
    layers.Dense(128, activation='relu'),

    # Sortie : 10 neurones = 1 par chiffre, softmax -> probabilites qui somment a 1
    # ex: 92% que c'est un 3, 5% un 8, 3% le reste
    layers.Dense(10, activation='softmax')
])

# Compilation
# optimizer='adam'              : descente de gradient adaptative, standard
# sparse_categorical_crossentropy: perte pour classification multi-classe
# avec labels entiers (0 a 9), pas besoin de one-hot encoding
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

t0 = time.time()

# Entrainement
# epochs=5         : le modele parcourt 5 fois l'integralite du dataset
# batch_size=128   : met a jour les poids tous les 128 exemples
# validation_split : 10% du train surveille la generalisation a chaque epoch
# Pourquoi : si val_loss remonte quand train_loss baisse -> surapprentissage
history = model.fit(x_train_cnn, y_train, epochs=5, batch_size=128,
                    validation_split=0.1, verbose=1)
train_time_cnn = time.time() - t0

# Evaluation sur le test set
_, acc_cnn = model.evaluate(x_test_cnn, y_test, verbose=0)

# argmax : prend l'indice du chiffre avec la probabilite la plus haute
y_pred_cnn = np.argmax(model.predict(x_test_cnn, verbose=0), axis=1)

print(f'Accuracy : {acc_cnn:.4f}')
print(f'Temps entrainement : {train_time_cnn:.1f}s')
print()
print(classification_report(y_test, y_pred_cnn))

# Courbes d'apprentissage : accuracy et loss sur train vs validation
# Pourquoi les deux : si val stagne alors que train monte -> overfitting
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy par epoch')
axes[0].legend()
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss par epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

# Matrice de confusion du CNN
cm = confusion_matrix(y_test, y_pred_cnn)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm).plot(ax=ax, colorbar=False)
ax.set_title('Matrice de confusion - CNN')
plt.tight_layout()
plt.show()

## Analyse Comparative

In [ ]:
# =============================================================
# OBJECTIF : Comparer les deux modeles selon 3 axes :
# 1. Performance  : accuracy globale et F1-score par chiffre
# 2. Complexite   : nombre de parametres appris et temps d'entrainement
# 3. Erreurs      : quelles images les deux modeles ratent en meme temps
# On cherche a repondre : le gain du CNN justifie-t-il sa complexite ?
# =============================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
import pandas as pd

# Rapports sous forme de dictionnaire pour extraire les metriques par chiffre
# Pourquoi output_dict=True : permet d'indexer par chiffre ("0", "1", etc.)
report_lr  = classification_report(y_test, y_pred_lr,  output_dict=True)
report_cnn = classification_report(y_test, y_pred_cnn, output_dict=True)

# Tableau recapitulatif des metriques cles
# Parametres LR  = taille de la matrice de poids + biais (784*10 + 10)
# Parametres CNN = tous les poids du reseau (filtres + couches Dense)
# Pourquoi comparer : un modele 100x plus complexe doit justifier son gain
summary = pd.DataFrame({
    'Modele':     ['Regression Logistique', 'CNN'],
    'Accuracy':   [acc_lr, acc_cnn],
    'Parametres': [logreg.coef_.size + logreg.intercept_.size, model.count_params()],
    'Temps (s)':  [round(train_time_lr, 1), round(train_time_cnn, 1)],
})
print(summary.to_string(index=False))
print()

# F1-score par chiffre : combine precision et rappel
# Pourquoi pas juste l'accuracy : certains chiffres sont difficiles (4 vs 9)
# le F1 montre precisement ou chaque modele echoue
digits = [str(i) for i in range(10)]
f1_lr  = [report_lr[d]['f1-score']  for d in digits]
f1_cnn = [report_cnn[d]['f1-score'] for d in digits]

# Graphe en barres cote a cote LR vs CNN pour chaque chiffre
# Pourquoi : voir sur quels chiffres le CNN apporte le plus de gain
x = np.arange(10)
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, f1_lr,  width, label='Regression Logistique')
ax.bar(x + width/2, f1_cnn, width, label='CNN')
ax.set_xticks(x)
ax.set_xticklabels([f'Chiffre {i}' for i in range(10)], rotation=45)
ax.set_ylabel('F1-score')
ax.set_title('F1-score par chiffre - LR vs CNN')
ax.set_ylim(0.9, 1.0)
ax.legend()
plt.tight_layout()
plt.show()

# np.where retourne les indices des images mal classees par chaque modele
errors_lr   = set(np.where(y_pred_lr  != y_test)[0])
errors_cnn  = set(np.where(y_pred_cnn != y_test)[0])

# Intersection : images ratees par les DEUX modeles en meme temps
# Pourquoi : ce sont les cas vraiment ambigus, pas des erreurs de modele
# mais des images difficiles meme pour un humain (mal ecrites, atypiques)
errors_both = errors_lr & errors_cnn

print(f'Erreurs LR  : {len(errors_lr)}')
print(f'Erreurs CNN : {len(errors_cnn)}')
print(f'Erreurs communes : {len(errors_both)}')
print()

# Affichage de 10 exemples difficiles avec les deux predictions
# Format titre : vrai label / prediction LR / prediction CNN
# Pourquoi : comprendre visuellement pourquoi les deux modeles se trompent
sample = list(errors_both)[:10]
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Exemples difficiles (rates par les deux modeles)', fontsize=13)
for ax, idx in zip(axes.flat, sample):
    ax.imshow(x_test[idx], cmap='gray')
    ax.set_title(f'Vrai:{y_test[idx]} LR:{y_pred_lr[idx]} CNN:{y_pred_cnn[idx]}', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()